# YOLO26n Student Baseline Seed 42 — Shared KD W&B Project

Notebook ini khusus **training student baseline YOLO26n tanpa knowledge distillation (seed 42)**. Evaluasi skripsi dipisahkan ke notebook berikutnya agar training tidak menahan objek evaluator/prediction di GPU.

Perbaikan utama dibanding notebook sebelumnya:

- memakai **integrasi W&B native Ultralytics**, bukan callback manual;
- `train/* loss`, `val/* loss`, Precision, Recall, mAP, learning rate, plots, model info, dan best-model artifact dicatat oleh integrasi native;
- W&B juga mencatat system metrics CPU/GPU secara otomatis;
- tidak memakai `wandb.watch()` karena histogram gradient/parameter tidak diperlukan untuk rumusan masalah dan menambah overhead;
- `cache=False`, AMP aktif, evaluasi berat dipisah, dan output checkpoint periodik dinonaktifkan untuk menekan RAM/VRAM serta ukuran folder;
- batch **tetap 16** agar konsisten dengan eksperimen sebelumnya. Jika Ultralytics melakukan auto-recovery OOM dan menurunkan batch, notebook sengaja menghentikan run agar eksperimen tidak diam-diam berubah.

> Notebook ini tetap menggunakan versi package dan hyperparameter dari pipeline terakhir agar seed 42 dapat dibandingkan dengan seed lain.

> **Organisasi eksperimen KD:** run ini dicatat pada project W&B `skripsi-yolo26-kd-indotraffic` bersama teacher YOLO26s dan seluruh run KD berikutnya. Baseline ini adalah kondisi kontrol `no-KD`; hyperparameter training tidak diubah dari baseline sebelumnya.


## 1. Instalasi dependensi yang dipin

In [ ]:
%pip install -q "ultralytics==8.4.155" "wandb==0.28.1" "roboflow==1.4.0" pyyaml pandas matplotlib pillow

# Penting: aktifkan setting SEBELUM Ultralytics/YOLO di-import,
# karena callback integration dibangun dari setting ini.
!yolo settings wandb=True


## 2. Environment, GPU, dan direktori

In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import sys
import time
import zipfile
from pathlib import Path
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import wandb
import yaml
from PIL import Image

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
PLATFORM = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"

if IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working/yolo26n_student_baseline_seed42")
elif IN_COLAB:
    WORK_ROOT = Path("/content/yolo26n_student_baseline_seed42")
else:
    WORK_ROOT = Path("./yolo26n_student_baseline_seed42")

WORK_ROOT = WORK_ROOT.resolve()
DATA_ROOT = WORK_ROOT / "dataset"
REPORTS_ROOT = WORK_ROOT / "reports"
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU tidak tersedia. Aktifkan GPU sebelum training.")

DEVICE = 0
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

ENV_INFO = {
    "platform": PLATFORM,
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "ultralytics": ultralytics.__version__,
    "wandb": wandb.__version__,
    "gpu": GPU_NAME,
    "gpu_vram_gb": round(GPU_VRAM_GB, 2),
}

print(json.dumps(ENV_INFO, indent=2))
print("WORK_ROOT:", WORK_ROOT)

## 3. Konfigurasi eksperimen — seed 42 saja

In [ ]:
CFG = {
    "seed": 42,

    # Dataset
    "workspace": "ilham-rafiqin",
    "project": "poltrada-original-raw-2",
    "dataset_version": 2,
    "class_names": [
        "Bus",
        "Mobil Penumpang",
        "Pejalan Kaki",
        "Sepeda Motor",
        "Truck",
        "Unmotorized",
    ],

    # Baseline & training — dipertahankan dari notebook sebelumnya
    "model": "yolo26n.pt",
    "role": "student_baseline",
    "treatment": "no_kd",
    "imgsz": 640,
    "batch": 16,
    "epochs": 100,
    "patience": 25,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "amp": True,
    "workers": 2,          # lebih konservatif untuk RAM notebook
    "cache": False,        # hindari host-RAM OOM
    "close_mosaic": 10,
    "flipud": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,

    # Checkpoint/output
    "save_period": -1,     # best.pt + last.pt; tidak membuat epoch1.pt ... epoch100.pt
    "plots": True,

    # W&B
    "wandb_project": "skripsi-yolo26-kd-indotraffic",
    "wandb_group": "student-baseline",
    "run_name": "yolo26n-student-baseline-seed42-v1",
}

EXPECTED_SPLITS = {
    "train": {"images": 6775, "labels": 6775, "annotations": 125108, "empty": 16},
    "val":   {"images": 1790, "labels": 1790, "annotations": 32076,  "empty": 2},
    "test":  {"images": 913,  "labels": 913,  "annotations": 16158,  "empty": 9},
}

print(json.dumps(CFG, indent=2))

### W&B experiment organization

Semua eksperimen knowledge distillation disimpan pada satu project W&B: `skripsi-yolo26-kd-indotraffic`. Run ini adalah **student baseline / no-KD control**. Teacher YOLO26s dan run Native KD, FGD, TinyKD, serta DCSF-KD dapat ditempatkan pada project yang sama dengan `group` berbeda agar perbandingan dashboard lebih mudah.

Perubahan pada notebook ini hanya menyangkut **organisasi eksperimen dan metadata W&B**. Dataset, model student, seed, batch size, optimizer, learning rate, augmentasi, epoch policy, dan parameter training lain tetap dipertahankan agar baseline tetap comparable.


## 4. Secrets dan aktivasi W&B native Ultralytics

In [ ]:
def get_secret(name):
    value = os.getenv(name)
    if value:
        return value

    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass

    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            pass

    return None

ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY")
WANDB_API_KEY = get_secret("WANDB_API_KEY")

if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY tidak ditemukan.")
if not WANDB_API_KEY:
    raise RuntimeError("WANDB_API_KEY tidak ditemukan. Run research ini wajib tercatat di W&B.")

wandb.login(key=WANDB_API_KEY, relogin=True)

# Metadata W&B tambahan. Ultralytics sendiri akan membuat/menutup run training.
os.environ["WANDB_RUN_GROUP"] = CFG["wandb_group"]
os.environ["WANDB_JOB_TYPE"] = "student-baseline"

from ultralytics import YOLO, settings

# PERUBAHAN PENTING dari notebook lama:
# integrasi W&B native sudah diaktifkan sebelum import melalui `yolo settings wandb=True`.
# Baris berikut menjadi hard gate agar run tidak berjalan tanpa logger native.
if settings.get("wandb") is not True:
    raise RuntimeError("Ultralytics W&B integration belum aktif.")

print("Ultralytics W&B setting:", settings.get("wandb"))

## 5. Download exact audited V2

Mekanisme download mengikuti cell yang sudah terbukti bekerja: workspace `ilham-rafiqin`, project `poltrada-original-raw-2`, version `2`, format `yolov8`.

In [ ]:
from roboflow import Roboflow

if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
rf_project = rf.workspace(CFG["workspace"]).project(CFG["project"])
downloaded = rf_project.version(CFG["dataset_version"]).download(
    "yolov8",
    location=str(DATA_ROOT),
    overwrite=True,
)

yaml_candidates = []
for root in [DATA_ROOT, Path(downloaded.location)]:
    if root.exists():
        yaml_candidates.extend(root.rglob("data.yaml"))

yaml_candidates = list(dict.fromkeys(p.resolve() for p in yaml_candidates))
if not yaml_candidates:
    raise FileNotFoundError("data.yaml tidak ditemukan setelah download.")

SOURCE_YAML = yaml_candidates[0]
raw_cfg = yaml.safe_load(SOURCE_YAML.read_text(encoding="utf-8"))

names = raw_cfg.get("names", [])
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]
else:
    names = list(names)

if names != CFG["class_names"]:
    raise ValueError(f"Class order berbeda.\nExpected: {CFG['class_names']}\nFound: {names}")

def resolve_split(key, aliases=()):
    value = raw_cfg.get(key)
    candidates = []
    if value:
        p = Path(str(value))
        candidates.append(p if p.is_absolute() else (SOURCE_YAML.parent / p).resolve())
    for alias in aliases:
        candidates.append((SOURCE_YAML.parent / alias / "images").resolve())
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Split {key} tidak ditemukan. Candidates: {candidates}")

SPLIT_IMAGES = {
    "train": resolve_split("train", ("train",)),
    "val": resolve_split("val", ("valid", "val")),
    "test": resolve_split("test", ("test",)),
}

print("SOURCE_YAML:", SOURCE_YAML)
for split, path in SPLIT_IMAGES.items():
    print(f"{split:5s}:", path)

## 6. Normalisasi polygon → enclosing bounding box

Roboflow V2 masih mengandung beberapa baris polygon. Karena tugas penelitian adalah object detection, polygon dikonversi **deterministik** menjadi enclosing bounding box YOLO. Bbox 5-kolom yang sudah ada tidak diubah. Backup label asli disimpan lokal.

In [ ]:
BACKUP_ROOT = REPORTS_ROOT / "labels_before_polygon_to_bbox"
if BACKUP_ROOT.exists():
    shutil.rmtree(BACKUP_ROOT)
BACKUP_ROOT.mkdir(parents=True)

conversion_log = []

def polygon_to_yolo_bbox(parts):
    cls = int(float(parts[0]))
    coords = list(map(float, parts[1:]))
    if len(coords) < 6 or len(coords) % 2 != 0:
        raise ValueError("Polygon harus memiliki >=3 titik.")
    xs = coords[0::2]
    ys = coords[1::2]
    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    # Clip hanya hasil enclosing polygon ke domain normalisasi yang sah.
    x1, y1 = max(0.0, x1), max(0.0, y1)
    x2, y2 = min(1.0, x2), min(1.0, y2)

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Polygon menghasilkan bbox tanpa area.")

    xc = (x1 + x2) / 2
    yc = (y1 + y2) / 2
    w = x2 - x1
    h = y2 - y1
    return cls, xc, yc, w, h

for split, images_dir in SPLIT_IMAGES.items():
    labels_dir = images_dir.parent / "labels"
    split_backup = BACKUP_ROOT / split
    shutil.copytree(labels_dir, split_backup)

    for label_path in sorted(labels_dir.glob("*.txt")):
        raw = label_path.read_text(encoding="utf-8").strip()
        if not raw:
            continue

        output = []
        changed = False

        for line_no, line in enumerate(raw.splitlines(), start=1):
            parts = line.split()

            if len(parts) == 5:
                # Detection bbox: pertahankan persis.
                output.append(line)
                continue

            if len(parts) > 5 and (len(parts) - 1) % 2 == 0:
                cls, xc, yc, w, h = polygon_to_yolo_bbox(parts)
                new_line = f"{cls} {xc:.10f} {yc:.10f} {w:.10f} {h:.10f}"
                output.append(new_line)
                conversion_log.append({
                    "split": split,
                    "label": label_path.name,
                    "line": line_no,
                    "original_tokens": len(parts),
                    "converted": new_line,
                })
                changed = True
                continue

            raise ValueError(
                f"Format anotasi tidak dikenali: {label_path}, line {line_no}: {line}"
            )

        if changed:
            label_path.write_text("\n".join(output) + "\n", encoding="utf-8")

conversion_df = pd.DataFrame(conversion_log)
display(conversion_df.head(20))
print("Polygon annotations converted:", len(conversion_df))

if len(conversion_df):
    conversion_df.to_csv(REPORTS_ROOT / "polygon_to_bbox_conversion.csv", index=False)

## 7. Dataset identity gate setelah normalisasi

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

audit_rows = []
fingerprint_hasher = hashlib.sha256()

for split, images_dir in SPLIT_IMAGES.items():
    labels_dir = images_dir.parent / "labels"
    images = sorted(p for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    labels = sorted(labels_dir.glob("*.txt"))

    annotations = 0
    empty = 0
    malformed = 0
    polygon_remaining = 0

    for label_path in labels:
        text = label_path.read_text(encoding="utf-8").strip()
        fingerprint_hasher.update(split.encode())
        fingerprint_hasher.update(label_path.name.encode())
        fingerprint_hasher.update(label_path.read_bytes())

        if not text:
            empty += 1
            continue

        for line in text.splitlines():
            parts = line.split()
            annotations += 1

            if len(parts) > 5:
                polygon_remaining += 1
                malformed += 1
                continue

            if len(parts) != 5:
                malformed += 1
                continue

            try:
                cls = int(float(parts[0]))
                xc, yc, w, h = map(float, parts[1:])
            except ValueError:
                malformed += 1
                continue

            if not (0 <= cls < len(names)):
                malformed += 1
            elif not all(np.isfinite([xc, yc, w, h])):
                malformed += 1
            elif w <= 0 or h <= 0:
                malformed += 1

    expected = EXPECTED_SPLITS[split]

    row = {
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "annotations": annotations,
        "empty": empty,
        "malformed_format": malformed,
        "polygon_remaining": polygon_remaining,
    }
    audit_rows.append(row)

    assert len(images) == expected["images"], (split, "images", len(images), expected["images"])
    assert len(labels) == expected["labels"], (split, "labels", len(labels), expected["labels"])
    assert annotations == expected["annotations"], (split, "annotations", annotations, expected["annotations"])
    assert empty == expected["empty"], (split, "empty", empty, expected["empty"])
    assert malformed == 0, (split, "malformed", malformed)
    assert polygon_remaining == 0, (split, "polygon_remaining", polygon_remaining)

AUDIT_DF = pd.DataFrame(audit_rows)
DATASET_FINGERPRINT = fingerprint_hasher.hexdigest()

display(AUDIT_DF)
print("Normalized-label fingerprint:", DATASET_FINGERPRINT)
print("DATASET IDENTITY GATE: PASSED")

AUDIT_DF.to_csv(REPORTS_ROOT / "dataset_identity_gate.csv", index=False)

## 8. Runtime YAML

In [ ]:
runtime_cfg = {
    "path": str(SOURCE_YAML.parent.resolve()),
    "train": str(SPLIT_IMAGES["train"]),
    "val": str(SPLIT_IMAGES["val"]),
    "test": str(SPLIT_IMAGES["test"]),
    "nc": len(names),
    "names": names,
}

RUNTIME_YAML = WORK_ROOT / "data_runtime.yaml"
RUNTIME_YAML.write_text(
    yaml.safe_dump(runtime_cfg, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print(RUNTIME_YAML.read_text(encoding="utf-8"))

## 9. OOM/reproducibility safeguards

- `batch=16` tetap, karena baseline seed lain menggunakan batch yang sama.
- Ultralytics versi baru dapat mencoba menurunkan batch otomatis bila OOM pada epoch pertama. Untuk penelitian komparatif, perubahan diam-diam itu tidak diizinkan: callback di bawah menghentikan run bila `trainer.batch_size` berubah.
- `AMP=True` mengurangi kebutuhan VRAM.
- `cache=False` menghindari penumpukan dataset di RAM.
- Tidak ada evaluasi COCO, inference gallery, atau benchmarking di notebook training ini.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def release_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

set_seed(CFG["seed"])
release_gpu_memory()
torch.cuda.reset_peak_memory_stats()

RUN_META = {
    "wandb_path": None,
    "wandb_url": None,
    "actual_batch": None,
}

def on_train_start_research(trainer):
    RUN_META["actual_batch"] = int(trainer.batch_size)

    # W&B native sudah diinisialisasi oleh callback Ultralytics sebelum training.
    if wandb.run is not None:
        RUN_META["wandb_path"] = wandb.run.path
        RUN_META["wandb_url"] = wandb.run.url
        wandb.run.config.update(
            {
                "research/role": "student_baseline",
                "research/experiment_family": "knowledge_distillation",
                "research/treatment": "no_kd",
                "research/student": "YOLO26n",
                "research/teacher": "none",
                "research/architecture": "YOLO26n native",
                "research/seed": CFG["seed"],
                "research/dataset_fingerprint": DATASET_FINGERPRINT,
                "research/dataset_version": CFG["dataset_version"],
                "research/expected_batch": CFG["batch"],
                "research/gpu": GPU_NAME,
            },
            allow_val_change=True,
        )
        print("W&B run:", wandb.run.url)

def enforce_fixed_batch(trainer):
    actual = int(trainer.batch_size)
    RUN_META["actual_batch"] = actual
    if actual != int(CFG["batch"]):
        raise RuntimeError(
            f"Batch berubah dari {CFG['batch']} menjadi {actual}. "
            "Kemungkinan terjadi OOM auto-recovery. Hentikan run ini dan pilih "
            "batch tetap yang lebih kecil untuk SEMUA konfigurasi penelitian."
        )

## 10. Training baseline seed 42

In [ ]:
RUN_NAME = CFG["run_name"]

# project dipakai Ultralytics sekaligus sebagai project W&B.
# Kita pindah cwd ke WORK_ROOT agar output lokal tetap terkumpul di satu folder.
os.chdir(WORK_ROOT)

EXPECTED_RUN_DIR = WORK_ROOT / CFG["wandb_project"] / RUN_NAME
if EXPECTED_RUN_DIR.exists():
    raise RuntimeError(
        f"Run directory sudah ada: {EXPECTED_RUN_DIR}\n"
        "Untuk fresh retraining seed 42, hapus/rename folder tersebut secara eksplisit "
        "sebelum menjalankan cell training."
    )

model = YOLO(CFG["model"])
model.add_callback("on_train_start", on_train_start_research)
model.add_callback("on_train_epoch_end", enforce_fixed_batch)

started = time.perf_counter()
status = "failed"
caught_error = None

try:
    results = model.train(
        data=str(RUNTIME_YAML),
        epochs=CFG["epochs"],
        imgsz=CFG["imgsz"],
        batch=CFG["batch"],
        device=DEVICE,
        workers=CFG["workers"],
        optimizer=CFG["optimizer"],
        lr0=CFG["lr0"],
        weight_decay=CFG["weight_decay"],
        warmup_epochs=CFG["warmup_epochs"],
        patience=CFG["patience"],
        amp=CFG["amp"],
        seed=CFG["seed"],
        deterministic=True,
        cache=CFG["cache"],
        flipud=CFG["flipud"],
        mixup=CFG["mixup"],
        cutmix=CFG["cutmix"],
        close_mosaic=CFG["close_mosaic"],
        save=True,
        save_period=CFG["save_period"],
        plots=CFG["plots"],
        val=True,
        project=CFG["wandb_project"],
        name=RUN_NAME,
        exist_ok=False,
        verbose=True,
    )
    status = "completed"

except Exception as exc:
    caught_error = exc
    raise

finally:
    elapsed = time.perf_counter() - started
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3

    TRAIN_META = {
        "status": status,
        "seed": CFG["seed"],
        "role": "student_baseline",
        "experiment_family": "knowledge_distillation",
        "treatment": "no_kd",
        "architecture": "YOLO26n native",
        "student": "YOLO26n",
        "teacher": None,
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "duration_seconds": elapsed,
        "duration_hours": elapsed / 3600,
        "peak_vram_gb": peak_vram_gb,
        "requested_batch": CFG["batch"],
        "actual_batch": RUN_META["actual_batch"],
        "wandb_path": RUN_META["wandb_path"],
        "wandb_url": RUN_META["wandb_url"],
        "error": None if caught_error is None else f"{type(caught_error).__name__}: {caught_error}",
        "environment": ENV_INFO,
        "config": CFG,
    }

    (REPORTS_ROOT / "training_metadata.json").write_text(
        json.dumps(TRAIN_META, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(TRAIN_META, indent=2))
    release_gpu_memory()

## 11. Training-output gate dan kelengkapan metrik

`results.csv` adalah salinan lokal authoritative dari history training. Cell ini memastikan ada empat kelompok informasi utama: **train loss, validation loss/metrics, detection metrics, dan learning rate**. Ini sekaligus memudahkan pengecekan bila dashboard W&B bermasalah.

In [ ]:
RUN_DIR = WORK_ROOT / CFG["wandb_project"] / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"
ARGS_YAML = RUN_DIR / "args.yaml"

required_files = [BEST_PT, LAST_PT, RESULTS_CSV, ARGS_YAML]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise RuntimeError("Output training tidak lengkap:\n" + "\n".join(missing))

history = pd.read_csv(RESULTS_CSV)
history.columns = [c.strip() for c in history.columns]

groups = {
    "train_losses": [c for c in history.columns if c.startswith("train/")],
    "val_losses": [c for c in history.columns if c.startswith("val/")],
    "metrics": [c for c in history.columns if c.startswith("metrics/")],
    "learning_rates": [c for c in history.columns if c.startswith("lr/")],
}

for group, cols in groups.items():
    print(f"{group:15s}: {cols}")
    if not cols:
        raise RuntimeError(f"Kelompok metrik '{group}' tidak ditemukan di results.csv")

display(history.tail())

map_col = next((c for c in history.columns if "mAP50-95" in c), None)
if map_col:
    best_idx = history[map_col].astype(float).idxmax()
    print("\nBest epoch row:")
    display(history.loc[[best_idx]])

print("\nTRAINING METRIC GATE: PASSED")
print("best.pt:", BEST_PT)
print("W&B    :", RUN_META.get("wandb_url"))

## 12. Visualisasi kurva training dari results.csv

In [ ]:
# Visualisasi lokal; W&B native sudah menyimpan history yang sama per epoch.
plot_cols = (
    groups["train_losses"]
    + groups["val_losses"]
    + [c for c in groups["metrics"] if "mAP" in c or "precision" in c or "recall" in c]
)

for col in plot_cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.arange(1, len(history) + 1), history[col])
    ax.set_xlabel("Epoch")
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.grid(alpha=0.25)
    plt.show()
    plt.close(fig)

## 13. Optional: verifikasi remote W&B

Cell ini tidak membatalkan hasil lokal bila API W&B sedang bermasalah. Tujuannya hanya memastikan run remote dapat ditemukan setelah sinkronisasi.

In [ ]:
if RUN_META.get("wandb_path"):
    try:
        api = wandb.Api(timeout=60)
        remote_run = api.run(RUN_META["wandb_path"])
        sample = remote_run.history(samples=5, pandas=True)
        print("Remote W&B run:", remote_run.url)
        print("Contoh keys remote:")
        print(sorted(sample.columns.tolist()))
    except Exception as exc:
        print("[WARN] Verifikasi remote W&B gagal:", type(exc).__name__, exc)
else:
    print("[WARN] W&B path tidak tertangkap.")

## 14. Paket download kecil

Hanya file yang benar-benar berguna untuk reproduksibilitas/analisis yang dipaketkan: `best.pt`, `last.pt`, `results.csv`, `args.yaml`, plot hasil training, metadata, dataset gate, dan conversion log. Folder dataset dan cache W&B tidak dimasukkan.

In [ ]:
BUNDLE_DIR = WORK_ROOT / "student_baseline_training_bundle_seed42"
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir(parents=True)

def copy_if_exists(src, dst_name=None):
    src = Path(src)
    if src.exists():
        dst = BUNDLE_DIR / (dst_name or src.name)
        if src.is_file():
            shutil.copy2(src, dst)

for src in [BEST_PT, LAST_PT, RESULTS_CSV, ARGS_YAML]:
    copy_if_exists(src)

for src in RUN_DIR.glob("*.png"):
    copy_if_exists(src)

for src in REPORTS_ROOT.glob("*"):
    if src.is_file():
        copy_if_exists(src, f"report_{src.name}")

manifest = []
for p in sorted(BUNDLE_DIR.glob("*")):
    if p.is_file():
        manifest.append({
            "file": p.name,
            "bytes": p.stat().st_size,
            "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        })

(BUNDLE_DIR / "MANIFEST.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

ZIP_PATH = WORK_ROOT / "yolo26n_student_baseline_seed42_training_bundle.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(BUNDLE_DIR.glob("*")):
        if p.is_file():
            zf.write(p, arcname=p.name)

print("Bundle:", ZIP_PATH)
print("Size MB:", round(ZIP_PATH.stat().st_size / 1024**2, 2))

## 15. Final gate

In [ ]:
assert status == "completed"
assert BEST_PT.exists()
assert LAST_PT.exists()
assert RESULTS_CSV.exists()
assert RUN_META["actual_batch"] == CFG["batch"]

print("=" * 72)
print("STUDENT BASELINE SEED 42 TRAINING: PASSED")
print("Run dir :", RUN_DIR)
print("best.pt :", BEST_PT)
print("W&B URL :", RUN_META.get("wandb_url"))
print("Bundle  :", ZIP_PATH)
print("=" * 72)